# LRS2.0 — YOLOv8s Training on Kaggle GPU

**Before running:**
1. Upload this notebook to kaggle.com → New Notebook → Upload
2. On the right panel: Accelerator → **GPU T4 x2** (or T4 x1)
3. Also on the right panel: Add your dataset via **+ Add Data** → upload the zip (see Cell 1)
4. Run cells top to bottom
5. The final cell downloads `best.pt` directly — place it at `models/plate_detector.pt`

In [ ]:
import os, zipfile, shutil

def extract_zip(zip_path, dest):
    print(f'Extracting: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)

# Find all zips under /kaggle/input
input_dir = '/kaggle/input'
zips = []
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith('.zip'):
            zips.append(os.path.join(root, f))

if zips:
    outer_zip = zips[0]
    extract_zip(outer_zip, '/kaggle/working/')

    # Handle double-zip: if extraction produced another zip, extract that too
    inner_zips = []
    for root, dirs, files in os.walk('/kaggle/working'):
        for f in files:
            if f.endswith('.zip'):
                inner_zips.append(os.path.join(root, f))

    for inner_zip in inner_zips:
        extract_zip(inner_zip, '/kaggle/working/')
        os.remove(inner_zip)

    print('Done. Contents of /kaggle/working:')
    print(os.listdir('/kaggle/working'))
else:
    # No zip — dataset was added directly via Kaggle "Add Data".
    # Find the root that contains dataset/data.yaml and copy it to /kaggle/working/
    print('No zip found. Searching for dataset/data.yaml in /kaggle/input ...')
    dataset_root = None
    for root, dirs, files in os.walk(input_dir):
        if 'data.yaml' in files and os.path.basename(root) == 'dataset':
            dataset_root = os.path.dirname(root)  # one level up from 'dataset/'
            break

    if dataset_root:
        dest = '/kaggle/working'
        print(f'Found dataset root: {dataset_root}')
        print('Copying to /kaggle/working/ ...')
        for item in os.listdir(dataset_root):
            src = os.path.join(dataset_root, item)
            dst = os.path.join(dest, item)
            if os.path.isdir(src):
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
        print('Done. Contents of /kaggle/working:')
        print(os.listdir('/kaggle/working'))
    else:
        print('ERROR: Could not find dataset/data.yaml anywhere under /kaggle/input')
        print('Contents of /kaggle/input:')
        for root, dirs, files in os.walk(input_dir):
            for f in files:
                print(os.path.join(root, f))


In [ ]:
# 2. Set working directory and verify structure
import os

# Try common extraction paths
for candidate in ['/kaggle/working/LRS2.0', '/kaggle/working']:
    if os.path.exists(os.path.join(candidate, 'dataset/data.yaml')):
        os.chdir(candidate)
        break

print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

In [ ]:
# 3. Install dependencies
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)
print('Done.')

In [ ]:
# 4. Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 5. Train YOLO11s @ 640px (~20-30 min on T4)
# Note: 1280px OOMs on T4 (14.5 GB) even at batch=2. 640px is standard for YOLOv8/11
# and works well for license plate detection.
!python scripts/train_yolo.py \
    --model yolo11s.pt \
    --epochs 200 \
    --patience 50 \
    --img 640 \
    --mosaic 0.8 \
    --device cuda

In [ ]:
# 6. Copy best.pt to a known output location
import glob, shutil, os

os.chdir('/kaggle/working')

candidates = sorted(glob.glob('runs/detect/train*/weights/best.pt'))
if candidates:
    best = candidates[-1]
    os.makedirs('models', exist_ok=True)
    shutil.copy(best, 'models/plate_detector.pt')
    size_mb = os.path.getsize('models/plate_detector.pt') / 1e6
    print(f'Copied {best} → models/plate_detector.pt ({size_mb:.1f} MB)')
else:
    print('No best.pt found — check training output above.')
    print('Current dir:', os.getcwd())
    print('Contents:', os.listdir('.'))

In [ ]:
# 7. Validate and print results
from ultralytics import YOLO
model = YOLO('models/plate_detector.pt')
metrics = model.val(data='dataset/data.yaml', device='cuda')
print(f'\n=== RESULTS ===')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

In [ ]:
# 8. Download best.pt to your computer
# Kaggle automatically saves everything in /kaggle/working/ as downloadable output.
# After the notebook finishes:
#   → Click the Output tab (right panel) → find models/plate_detector.pt → Download
# OR use this cell to copy it to the output root for easier access:
import shutil
shutil.copy('models/plate_detector.pt', '/kaggle/working/plate_detector.pt')
print('plate_detector.pt is ready to download from the Output tab.')